In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

print("pandas :", pd.__version__)
print("numpy  :", np.__version__)

pandas : 2.3.3
numpy  : 1.26.4


In [2]:
RAW_DIR = Path("../../data/raw")

year_files = {
    2020: RAW_DIR / "NO2_2020.csv",
    2021: RAW_DIR / "NO2_2021.csv",
    2022: RAW_DIR / "NO2_2022.csv",
    2023: RAW_DIR / "NO2_2023.csv",
}

df_list = []
for year, fpath in year_files.items():
    temp = pd.read_csv(fpath, skiprows=7, low_memory=False, encoding="utf-8-sig")
    temp.columns = temp.columns.str.strip()
    temp["_source_year"] = year  
    temp["Date//Date"] = pd.to_datetime(temp["Date//Date"], errors="coerce")
    df_list.append(temp)
    print(f"NO2_{year}.csv → {len(temp):>6,} rows | date sample: {temp['Date//Date'].iloc[0]!r}")

df = pd.concat(df_list, ignore_index=True)
print(f"Combined shape: {df.shape}")


NO2_2020.csv → 71,736 rows | date sample: Timestamp('2020-01-01 00:00:00')
NO2_2021.csv → 71,905 rows | date sample: Timestamp('2021-01-01 00:00:00')
NO2_2022.csv → 71,540 rows | date sample: Timestamp('2022-01-01 00:00:00')
NO2_2023.csv → 71,540 rows | date sample: Timestamp('2023-01-01 00:00:00')
Combined shape: (286721, 32)


In [3]:
hour_cols = [c for c in df.columns if re.match(r"^H\d{2}", str(c))]
hour_cols = sorted(hour_cols)

if len(hour_cols) == 0:
    raise ValueError("No hourly columns found (expected H01..H24). Check the input files/headers.")

print(f"Hourly columns found: {len(hour_cols)} ({hour_cols[0]} … {hour_cols[-1]})")

for c in hour_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df[hour_cols] = df[hour_cols].replace(-999, np.nan)

n_sentinel = (df[hour_cols].isna()).sum().sum()  
print("Hourly columns converted to numeric and sentinel -999 handled.")


Hourly columns found: 24 (H01//H01 … H24//H24)
Hourly columns converted to numeric and sentinel -999 handled.


In [4]:
df["NO2_daily"] = df[hour_cols].mean(axis=1, skipna=True)
df["valid_hours"] = df[hour_cols].notna().sum(axis=1)

print(df[["NO2_daily", "valid_hours"]].describe().round(3))


        NO2_daily  valid_hours
count  279486.000   286721.000
mean        5.953       22.802
std         5.389        3.959
min         0.000        0.000
25%         2.167       23.000
50%         4.375       24.000
75%         8.083       24.000
max        58.000       24.000


In [5]:
final = df[["Date//Date", "City//Ville", "NO2_daily", "valid_hours"]].copy()
final = final.rename(columns={"Date//Date": "Date", "City//Ville": "City"})

before = len(final)
final = final.dropna(subset=["NO2_daily"])
print(f"Rows dropped (missing NO2_daily): {before - len(final):,}")

before = len(final)
final = final.drop_duplicates(subset=["City", "Date"])
print(f"Duplicate City-Date rows removed: {before - len(final):,}")

print("Columns currently in final:", final.columns.tolist())
print("Shape:", final.shape)


Rows dropped (missing NO2_daily): 7,235
Duplicate City-Date rows removed: 46,534
Columns currently in final: ['Date', 'City', 'NO2_daily', 'valid_hours']
Shape: (232952, 4)


In [6]:
date_str = final["Date"].astype(str).str.strip()

looks_like_date = (
    date_str.str.match(r"^\d{4}-\d{2}-\d{2}$") |   
    date_str.str.match(r"^\d{1,2}/\d{1,2}/\d{4}$") 
)

print("Rows with weird Date strings:", (~looks_like_date).sum())
print(date_str[~looks_like_date].head(10))

final = final[looks_like_date].copy()

Rows with weird Date strings: 0
Series([], Name: Date, dtype: object)


In [7]:
date_str = final["Date"].astype(str).str.strip()

final["Date"] = pd.to_datetime(date_str, format="%Y-%m-%d", errors="coerce")

mask = final["Date"].isna()
final.loc[mask, "Date"] = pd.to_datetime(date_str[mask], format="%m/%d/%Y", errors="coerce")

bad_dates = final["Date"].isna().sum()
print(f"Unparsed Date rows (NaT): {bad_dates:,}")

if bad_dates > 0:
    mask = final["Date"].isna()
    final.loc[mask, "Date"] = pd.to_datetime(date_str[mask], errors="coerce")

print("Date range:", final["Date"].min(), "→", final["Date"].max())


Unparsed Date rows (NaT): 0
Date range: 2020-02-18 00:00:00 → 2216-05-28 00:00:00


In [8]:
df_check = final.copy()   

jan_mask = (df_check["Date"] >= "2020-01-01") & (df_check["Date"] <= "2020-01-31")
print("Rows in Jan 2020 (final):", jan_mask.sum())
print("Min date:", df_check["Date"].min(), "| Max date:", df_check["Date"].max())

Rows in Jan 2020 (final): 0
Min date: 2020-02-18 00:00:00 | Max date: 2216-05-28 00:00:00


In [9]:
pre = df.copy()

date_col = "Date//Date" if "Date//Date" in pre.columns else "Date"

jan = pre[(pre[date_col] >= "2020-01-01") & (pre[date_col] <= "2020-01-31")]

print("Jan 2020 rows (before dropping):", len(jan))
print("Jan 2020 NO2_daily missing:", jan["NO2_daily"].isna().sum())
print("Jan 2020 valid avg rows:", jan["NO2_daily"].notna().sum())

if "valid_hours" in jan.columns:
    print("\nValid hours summary in Jan 2020:")
    print(jan["valid_hours"].describe())
    print("Days with >=18 hours:", (jan["valid_hours"] >= 18).sum())
    print("Days with >=12 hours:", (jan["valid_hours"] >= 12).sum())

Jan 2020 rows (before dropping): 31
Jan 2020 NO2_daily missing: 31
Jan 2020 valid avg rows: 0

Valid hours summary in Jan 2020:
count    31.0
mean      0.0
std       0.0
min       0.0
25%       0.0
50%       0.0
75%       0.0
max       0.0
Name: valid_hours, dtype: float64
Days with >=18 hours: 0
Days with >=12 hours: 0


In [10]:
final["Year"] = final["Date"].dt.year
final["Month"] = final["Date"].dt.month

no2_cityday = final[["City", "Date","NO2_daily" ,"Year", "Month",]].copy()

no2_cityday = no2_cityday.sort_values(["City", "Date"]).reset_index(drop=True)

no2_cityday["Date"] = no2_cityday["Date"].dt.strftime("%Y-%m-%d")

print(no2_cityday.head())
print("NO2_cityday shape:", no2_cityday.shape)


      City        Date  NO2_daily  Year  Month
0  Agassiz  2021-01-01   7.208333  2021      1
1  Agassiz  2021-01-02   8.083333  2021      1
2  Agassiz  2021-01-03  11.916667  2021      1
3  Agassiz  2021-01-04   9.583333  2021      1
4  Agassiz  2021-01-05  12.750000  2021      1
NO2_cityday shape: (232952, 5)


In [11]:
print("=" * 55)
print("      NO₂ CITY-DAY — FINAL VALIDATION REPORT")
print("=" * 55)
print(f"Total records    : {len(no2_cityday):,}")
print(f"Unique cities    : {no2_cityday['City'].nunique()}")
print(f"Years captured   : {sorted(no2_cityday['Year'].unique())}")
print(f"Date range       : {no2_cityday['Date'].min()} → {no2_cityday['Date'].max()}")
print("\nMissing values:")
print(no2_cityday.isna().sum().to_string())


      NO₂ CITY-DAY — FINAL VALIDATION REPORT
Total records    : 232,952
Unique cities    : 156
Years captured   : [2020, 2021, 2022, 2023, 2024, 2025, 2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033, 2034, 2035, 2036, 2037, 2038, 2039, 2040, 2041, 2042, 2043, 2044, 2045, 2046, 2047, 2048, 2049, 2050, 2051, 2052, 2053, 2054, 2055, 2056, 2057, 2058, 2059, 2060, 2061, 2062, 2063, 2064, 2065, 2066, 2067, 2068, 2069, 2070, 2071, 2072, 2073, 2074, 2075, 2076, 2077, 2078, 2079, 2080, 2081, 2082, 2083, 2084, 2085, 2086, 2087, 2088, 2089, 2090, 2091, 2092, 2093, 2094, 2095, 2096, 2097, 2098, 2099, 2100, 2101, 2102, 2103, 2104, 2105, 2106, 2107, 2108, 2109, 2110, 2111, 2112, 2113, 2114, 2115, 2116, 2117, 2118, 2119, 2120, 2121, 2122, 2123, 2124, 2125, 2126, 2127, 2128, 2129, 2130, 2131, 2132, 2133, 2134, 2135, 2136, 2137, 2138, 2139, 2140, 2141, 2142, 2143, 2144, 2145, 2146, 2147, 2148, 2149, 2150, 2151, 2152, 2153, 2154, 2155, 2156, 2157, 2158, 2159, 2160, 2161, 2162, 2163, 2164, 2165, 2166, 216

In [12]:
OUT_DIR = Path("../../data/validated")
OUT_DIR.mkdir(parents=True, exist_ok=True)

out_path = OUT_DIR / "NO2_cityday.csv"
no2_cityday.to_csv(out_path, index=False)

print(f"Saved: {out_path}")
print(f"Shape  : {no2_cityday.shape}")
print(f"Cols   : {no2_cityday.columns.tolist()}")


Saved: ..\..\data\validated\NO2_cityday.csv
Shape  : (232952, 5)
Cols   : ['City', 'Date', 'NO2_daily', 'Year', 'Month']
